# Эксперименты: почему пайплайн выдаёт то, что выдаёт

Датасет **Music4All-CRS** (`docs/dataset.md`): у пользователя есть история прослушиваний Last.fm и текстовый профиль,
на каждый запрос — ровно один правильный трек. Метрика — nDCG@20.

Особенности, от которых зависит решение:
* 13 типов запросов (`query_type`): 11 к прослушанным в окне трекам (точное название, строчка текста, жанр,
  настроение, эпоха...) и два discovery — `novelty` (новый артист) и `similar_to` («like X by Y but from other artists»);
* **большинство целей уже есть в истории** (`is_new = False`), поэтому прослушанное по умолчанию не выкидываем;
  для `novelty` / `similar_to` — наоборот;
* запросы и теги каталога на английском.

![scheme](scheme.png)

| Шаг | Модуль | Сейчас |
|---|---|---|
| 1. разбор запроса | `recsys/dialog.py` | правила: теги, артисты, эпохи, страны, язык, образец X, «только новое»; LLM по флагу |
| 2. кандидаты | `recsys/retrieval/` | локально: relisten, title, history, audio (MuQ), popular, bm25; с сервера: BM25, HNSW |
| 3. RRF + фильтры + дедуп | `recsys/fusion.py` | настоящий |
| 4. ранкер | `recsys/ranking.py` | **заглушка** (порядок RRF), `heuristic`, клиент `api` |
| 5. описание от LLM | `recsys/explain.py` | **заглушка** (приветствие + список), LLM по флагу |

Код живёт в `.py`, ноутбук только импортирует и запускает. Здесь: разбор одного запроса по шагам, как правила
понимают настоящие запросы, сравнение вариантов конфига и LLM вместо правил.

* ответ на запрос — `inference.ipynb`
* метрики на валидации — `python evaluate.py` (`README.md`)
* проверка сервисов BM25 и HNSW руками — `examples/candgen.ipynb`

Установка зависимостей (быстро, если всё уже стоит).

In [ ]:
%pip install -q -r requirements.txt

Импорты. `autoreload` подхватывает правки в `.py` без перезапуска ядра.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from recsys.config import load_config
from recsys.data import load_data, history_frame
from recsys.pipeline import Pipeline
from recsys.schemas import Request
from recsys.eval import compare_configs

pd.set_option("display.max_colwidth", 80)

## Конфиг

Все настройки лежат в `configs/default.yaml`; нужное можно переопределить в `OVERRIDES`.

* `data.source`: `crs` — Music4All-CRS из `data.crs.dir` (по умолчанию `music4all_crs/`), `synthetic` — сгенерированные данные (только для проверки кода)
* `data.crs.n_users`: сколько пользователей брать из сплита (в test_public 12.2k), `data.max_queries_per_user`: сколько запросов на пользователя
* `retrieval.<источник>.enabled`, `ranker.type`, `summarizer.type` / `explainer.type` (`llm` — модель в DataSphere)
* `USE_CANDGEN = True`: BM25 и HNSW берутся по HTTP с сервера (адреса и ключи — в `.env` или секретах DataSphere, API — `docs/candgen_api.md`). Без флага работает локальный `bm25`, HNSW нет

In [ ]:
OVERRIDES = {
    # данные по умолчанию — music4all_crs/ (configs/default.yaml); другая папка или сколько пользователей:
    # "data": {"crs": {"dir": "music4all_crs", "n_users": 300}},
    # поиск по строчке текста песни:
    # "data": {"crs": {"with_lyrics": True}}, "retrieval": {"lyrics": {"enabled": True}},
    # "ranker": {"type": "heuristic"},
}
USE_CANDGEN = True  # True: BM25 и HNSW с сервера по HTTP (configs/candgen.yaml)

config_files = ["configs/default.yaml"] + (["configs/candgen.yaml"] if USE_CANDGEN else [])
cfg = load_config(config_files, OVERRIDES)
cfg["data"]["source"], cfg["candgen"]["bm25"]["url"] if USE_CANDGEN else "кандгены локально"

## Данные

Каталог треков и запросы по сплитам: один запрос = один позитив пользователя (`data.requests` — сплит для оценки, `data.eval_split`).

In [ ]:
data = load_data(cfg)
catalog = data.catalog
display(catalog.stats().to_frame("value").T)
catalog.df[["track_id", "artist", "title", "tags", "genres", "release_year", "lang", "popularity"]].head()

### Как выглядит запрос

Запрос пользователя, его тип, профиль, история (уникальные треки: сколько раз и когда последний раз) и цель — трек, который нужно найти.

In [ ]:
r = data.requests[0]
print("запрос:", r.dialog[0].text)
print("тип:", r.meta["query_type"], "| новый трек:", r.meta["is_new"], "| цель:", r.meta["artist"], "-", r.meta["title"])
print("профиль:", r.user_info[:300], "...")
print("демография:", r.user_attrs)
history_frame(r).head()

## Пайплайн

In [ ]:
pipe = Pipeline.from_config(cfg, catalog)
print("источники:", [r.name for r in pipe.retrievers])
print("шаги:", type(pipe.summarizer).__name__, type(pipe.ranker).__name__, type(pipe.explainer).__name__)
for x in pipe.retrievers:  # сервисы на сервере: доступность и версия индекса
    if getattr(x, "remote", False):
        print(f"  {x.name}:", x.health())

## Разбор одного запроса

На каком месте цель в выдаче и в каждом источнике кандидатов, веса RRF, время шагов и признаки, которые видит ранкер.
Если цели нет в выдаче, здесь видно, на каком шаге она потерялась: не нашёл ни один источник, отсеялась при слиянии
или ранкер поставил её ниже.

In [ ]:
resp = pipe.run(r, debug=True)
target = r.target_ids[0]
print("запрос:", r.dialog[-1].text, "| цель:", r.meta["artist"], "—", r.meta["title"])
print("место цели в выдаче:", resp.track_ids.index(target) + 1 if target in resp.track_ids else "нет в top-k")

In [ ]:
dbg = resp.debug
print("1. саммари:", resp.summary)
print("2. кандидатов по источникам:", dbg["n_candidates"])
print("   место цели в источниках:", {s: next((c.rank for c in cs if c.track_id == target), None)
                                    for s, cs in dbg["candidates"].items()})
print("3. веса RRF:", dbg["weights"], "| после фильтров:", dbg["n_fused"])
print("сервисы:", {x.name: x.last_error or "ok" for x in pipe.retrievers if getattr(x, "remote", False)})
print("время, мс:", {k: round(v * 1000, 1) for k, v in dbg["timings"].items()})
dbg["features"].head(10)

## Как правила понимают настоящие запросы

По 2 запроса каждого типа из `train` (папка `data.crs.dir`): какие теги, исключения, страну, язык и год вытащил шаг 1.
Ошибки здесь — повод поправить правила в `recsys/dialog.py`.

In [ ]:
from recsys.data.crs import read_split, requests_from_tables

try:
    train = read_split(cfg["data"]["crs"]["dir"], "train", n_users=300)
except FileNotFoundError:
    train = None
if train:
    real = requests_from_tables(train, catalog=catalog)
    rows = []
    for fam, group in pd.Series(real).groupby([q.meta["query_type"] for q in real]):
        for q in group.head(2):
            sm = pipe.summarizer.summarize(q)
            rows.append({"тип": fam, "запрос": q.dialog[0].text, "цель": f"{q.meta['artist']} — {q.meta['title']}",
                         "теги": sm.include_tags, "исключить": sm.exclude_tags,
                         "страна/язык/год": sm.countries + sm.languages + sm.years})
    display(pd.DataFrame(rows))
else:
    print("train не найден в data.crs.dir: положите туда файлы датасета")

## Сравнение вариантов

Одни и те же запросы, разные конфиги. Значение — nDCG@20 по типам запросов, строка ALL — в среднем.
Одиночный прогон с метриками и файлами результата — `python evaluate.py` (можно с `--set`, например `--set ranker.type=heuristic`).

На синтетике метрики ничего не говорят о реальном качестве: запросы построены по тем же тегам, по которым ищем.

In [ ]:
variants = {
    "stub ranker": {},
    "heuristic ranker": {"ranker": {"type": "heuristic"}},
    "без relisten": {"retrieval": {"relisten": {"enabled": False}}},
    "без title": {"retrieval": {"title": {"enabled": False}}},
    "без audio": {"retrieval": {"audio": {"enabled": False}}},
    "без hnsw": {"retrieval": {"hnsw": {"enabled": False}}},
    "без запроса (только история)": {"retrieval": {k: {"enabled": False} for k in
                                     ["bm25", "title", "bm25_cards", "hnsw"]}},
}
compare_configs(cfg, variants, catalog, data.requests[:400], by="query_type").round(2)

## LLM (саммари и описание)

Нужна GPU-конфигурация DataSphere и `transformers` (см. requirements.txt). Поставьте `USE_LLM = True`.
Модель загружается один раз и работает на шагах 1 и 5. Если LLM вернула не JSON, саммари берётся из правил.

In [ ]:
USE_LLM = False

if USE_LLM:
    from recsys.llm import build_llm
    cfg_llm = load_config(config_files, {
        **OVERRIDES,
        "llm": {"type": "local", "model_name": "Qwen/Qwen2.5-1.5B-Instruct"},
        "summarizer": {"type": "llm"},
        "explainer": {"type": "llm"},
    })
    llm = build_llm(cfg_llm)
    pipe_llm = Pipeline.from_config(cfg_llm, catalog, llm=llm, bm25_index=pipe.bm25_index)
    resp_llm = pipe_llm.run(r, debug=True)
    print("RAW LLM SUMMARY:", pipe_llm.summarizer.last_raw)
    print("SUMMARY:", resp_llm.summary)
    print(resp_llm.text)
    display(resp_llm.to_frame())

## Как заменить заглушку

* **Ранкер**: класс с `rank(features, ctx) -> DataFrame` (колонка `rank_score`) в `recsys/ranking.py`, признаки в `build_features` там же. Для внешнего сервиса: `ranker.type: api`, `ranker.api_url`.
* **Новый источник кандидатов**: класс-наследник `BaseRetriever` с `search(ctx) -> [Candidate]` в `recsys/retrieval/`, зарегистрировать в `build_retrievers` (`recsys/retrieval/sources.py`) и включить в `configs/default.yaml`.
* **Разбор запроса**: правила в `recsys/dialog.py` или `summarizer.type: llm`.